# 데이터 설계 진단

1. 날짜 alignment
- sleep target의 calendar_date가 sleep start date인지 end date인지
- activity/stress/HRV 날짜와 같은 의미인지
- 밤을跨는 sleep record가 어느 날짜에 붙었는지

2. label 기준
- good_sleep_label이 어떤 metric/threshold로 생성됐는지
- global threshold인지 participant-relative threshold인지
- label이 participant별로 치우치지 않는지

3. leakage
- target 생성에 쓰인 sleep outcome column이 feature에 남아 있지 않은지
- stress_sleep_points 같은 명시적 sleep feature 제외 여부
- HRV/SpO2/respiratory가 same-night sleep-derived인지

4. split 분포
- train/validation/test target rate
- feature mean/std
- missing rate
- participant 수와 row 수
- window sample 수

5. participant 차이
- participant별 row count
- participant별 positive rate
- participant별 window 기여도
- 특정 participant가 train/test 성능을 지배하는지

6. missingness
- missing indicator와 target 상관
- record_count와 target 상관
- split별 missingness 차이
- PCA PC1/PC2가 availability 구조였던 점 재검토

7. window 생성
- contiguous window 조건으로 제외된 sample/participant
- row-based rolling 대비 sample 수 차이
- padded + mask 방식 필요성

8. scaling
- global scaling으로 충분한지
- HRV/resting HR/activity 같은 개인차 feature를 participant-relative로 바꿔야 하는지

9. feature set
- all_features 대신 wearable_only, wearable_values_only, no_missing_indicators, no_availability_features 비교

10. label noise
- threshold 근처 애매한 label 비율
- 확실한 good/bad만 남겼을 때 성능
- regression target 가능성

## 1. 날짜 alignment

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")

daily_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_daily.csv"
target_path = PROJECT_ROOT / "data" / "processed" / "daily_aggregates" / "sleep_daily_target.csv"

daily = pd.read_csv(daily_path)
target = pd.read_csv(target_path)

print("daily shape:", daily.shape)
print("target shape:", target.shape)
print("\ndaily columns:")
print(daily.columns.tolist())
print("\ntarget columns:")
print(target.columns.tolist())

daily shape: (3551, 130)
target shape: (3551, 28)

daily columns:
['participant_object_id', 'calendar_date', 'mongo_doc_id', 'logId', 'startTime', 'endTime', 'minutesAsleep', 'minutesAwake', 'timeInBed', 'efficiency', 'sleep_duration_hours', 'time_in_bed_hours', 'awake_ratio', 'good_sleep_label', 'deep_minutes', 'light_minutes', 'rem_minutes', 'wake_minutes', 'deep_ratio', 'light_ratio', 'rem_ratio', 'wake_ratio', 'asleep_minutes', 'restless_minutes', 'awake_minutes', 'classic_asleep_ratio', 'classic_restless_ratio', 'classic_awake_ratio', 'stress_score_mean', 'stress_sleep_points_mean', 'stress_responsiveness_points_mean', 'stress_exertion_points_mean', 'stress_ready_rate', 'stress_calculation_failed_rate', 'stress_record_count', 'hrv_summary_rmssd_mean', 'hrv_summary_nremhr_mean', 'hrv_summary_entropy_mean', 'hrv_summary_record_count', 'hrv_detail_rmssd_mean', 'hrv_detail_rmssd_std', 'hrv_detail_rmssd_min', 'hrv_detail_rmssd_max', 'hrv_detail_coverage_mean', 'hrv_detail_coverage_std'

In [2]:
# 날짜 관련 컬럼 확인
date_like_cols = [
    col for col in target.columns
    if "date" in col.lower()
    or "time" in col.lower()
    or "start" in col.lower()
    or "end" in col.lower()
]

print("date/time-like columns in sleep target:")
for col in date_like_cols:
    print("-", col)

target[date_like_cols + ["participant_object_id", "calendar_date", "good_sleep_label"]].head(20)

date/time-like columns in sleep target:
- calendar_date
- startTime
- endTime
- timeInBed
- time_in_bed_hours


,calendar_date,startTime,endTime,timeInBed,time_in_bed_hours,participant_object_id,calendar_date,good_sleep_label
0,2021-05-24,2021-05-24 00:40:00,2021-05-24 09:21:00,521,8.683333,621e2e8e67b776a24055b564,2021-05-24,1
1,2021-05-25,2021-05-24 23:48:30,2021-05-25 08:56:30,548,9.133333,621e2e8e67b776a24055b564,2021-05-25,1
2,2021-05-26,2021-05-25 23:46:30,2021-05-26 09:06:30,560,9.333333,621e2e8e67b776a24055b564,2021-05-26,1
3,2021-05-27,2021-05-26 23:21:30,2021-05-27 09:48:30,627,10.450000,621e2e8e67b776a24055b564,2021-05-27,1
4,2021-05-28,2021-05-27 23:37:00,2021-05-28 08:58:30,561,9.350000,621e2e8e67b776a24055b564,2021-05-28,1
5,2021-05-29,2021-05-28 23:50:00,2021-05-29 08:48:00,538,8.966667,621e2e8e67b776a24055b564,2021-05-29,1
6,2021-05-30,2021-05-30 00:30:00,2021-05-30 09:45:30,555,9.250000,621e2e8e67b776a24055b564,2021-05-30,1
7,2021-05-31,2021-05-30 23:29:30,2021-05-31 09:08:30,579,9.650000,621e2e8e67b776a24055b564,2021-05-31,1
8,2021-06-01,2021-05-31 23:42:30,2021-06-01 09:26:30,584,9.733333,621e2e8e67b776a24055b564,2021-06-01,1
9,2021-06-02,2021-06-01 23:58:30,2021-06-02 09:32:30,574,9.566667,621e2e8e67b776a24055b564,2021-06-02,1


In [3]:
# start/end time이 있다면 calendar_date가 start 기준인지 end 기준인지 확인
check = target.copy()

for col in date_like_cols:
    check[col] = pd.to_datetime(check[col], errors="coerce")

if "startTime" in check.columns:
    check["start_date_from_startTime"] = check["startTime"].dt.date.astype("string")

if "endTime" in check.columns:
    check["end_date_from_endTime"] = check["endTime"].dt.date.astype("string")

check["calendar_date_str"] = pd.to_datetime(check["calendar_date"], errors="coerce").dt.date.astype("string")

compare_cols = ["participant_object_id", "calendar_date", "good_sleep_label"]

if "start_date_from_startTime" in check.columns:
    compare_cols.append("start_date_from_startTime")
    check["calendar_equals_start_date"] = check["calendar_date_str"] == check["start_date_from_startTime"]
    compare_cols.append("calendar_equals_start_date")

if "end_date_from_endTime" in check.columns:
    compare_cols.append("end_date_from_endTime")
    check["calendar_equals_end_date"] = check["calendar_date_str"] == check["end_date_from_endTime"]
    compare_cols.append("calendar_equals_end_date")

check[compare_cols].head(30)

,participant_object_id,calendar_date,good_sleep_label,start_date_from_startTime,calendar_equals_start_date,end_date_from_endTime,calendar_equals_end_date
0,621e2e8e67b776a24055b564,2021-05-24,1,2021-05-24,True,2021-05-24,True
1,621e2e8e67b776a24055b564,2021-05-25,1,2021-05-24,False,2021-05-25,True
2,621e2e8e67b776a24055b564,2021-05-26,1,2021-05-25,False,2021-05-26,True
3,621e2e8e67b776a24055b564,2021-05-27,1,2021-05-26,False,2021-05-27,True
4,621e2e8e67b776a24055b564,2021-05-28,1,2021-05-27,False,2021-05-28,True
5,621e2e8e67b776a24055b564,2021-05-29,1,2021-05-28,False,2021-05-29,True
6,621e2e8e67b776a24055b564,2021-05-30,1,2021-05-30,True,2021-05-30,True
7,621e2e8e67b776a24055b564,2021-05-31,1,2021-05-30,False,2021-05-31,True
8,621e2e8e67b776a24055b564,2021-06-01,1,2021-05-31,False,2021-06-01,True
9,621e2e8e67b776a24055b564,2021-06-02,1,2021-06-01,False,2021-06-02,True


In [4]:
# calendar_date가 start/end 중 어느 쪽과 더 많이 일치하는지 요약
summary = {}

if "calendar_equals_start_date" in check.columns:
    summary["calendar_equals_start_date_rate"] = check["calendar_equals_start_date"].mean()
    summary["calendar_equals_start_date_count"] = int(check["calendar_equals_start_date"].sum())

if "calendar_equals_end_date" in check.columns:
    summary["calendar_equals_end_date_rate"] = check["calendar_equals_end_date"].mean()
    summary["calendar_equals_end_date_count"] = int(check["calendar_equals_end_date"].sum())

summary["rows"] = len(check)

summary

{'calendar_equals_start_date_rate': np.float64(0.620107012109265),
 'calendar_equals_start_date_count': 2202,
 'calendar_equals_end_date_rate': np.float64(1.0),
 'calendar_equals_end_date_count': 3551,
 'rows': 3551}

In [5]:
# sleep이 자정을 넘는지 확인
if {"startTime", "endTime"}.issubset(check.columns):
    check["sleep_crosses_midnight"] = (
        check["startTime"].dt.date.astype("string")
        != check["endTime"].dt.date.astype("string")
    )

    print("cross-midnight rate:", check["sleep_crosses_midnight"].mean())
    print("cross-midnight count:", int(check["sleep_crosses_midnight"].sum()))
    
    display(
        check[
            [
                "participant_object_id",
                "calendar_date",
                "startTime",
                "endTime",
                "start_date_from_startTime",
                "end_date_from_endTime",
                "sleep_crosses_midnight",
                "good_sleep_label",
            ]
        ].head(30)
    )
else:
    print("startTime/endTime columns not found.")

cross-midnight rate: 0.379892987890735
cross-midnight count: 1349


,participant_object_id,calendar_date,startTime,endTime,start_date_from_startTime,end_date_from_endTime,sleep_crosses_midnight,good_sleep_label
0,621e2e8e67b776a24055b564,2021-05-24,2021-05-24 00:40:00,2021-05-24 09:21:00,2021-05-24,2021-05-24,False,1
1,621e2e8e67b776a24055b564,2021-05-25,2021-05-24 23:48:30,2021-05-25 08:56:30,2021-05-24,2021-05-25,True,1
2,621e2e8e67b776a24055b564,2021-05-26,2021-05-25 23:46:30,2021-05-26 09:06:30,2021-05-25,2021-05-26,True,1
3,621e2e8e67b776a24055b564,2021-05-27,2021-05-26 23:21:30,2021-05-27 09:48:30,2021-05-26,2021-05-27,True,1
4,621e2e8e67b776a24055b564,2021-05-28,2021-05-27 23:37:00,2021-05-28 08:58:30,2021-05-27,2021-05-28,True,1
5,621e2e8e67b776a24055b564,2021-05-29,2021-05-28 23:50:00,2021-05-29 08:48:00,2021-05-28,2021-05-29,True,1
6,621e2e8e67b776a24055b564,2021-05-30,2021-05-30 00:30:00,2021-05-30 09:45:30,2021-05-30,2021-05-30,False,1
7,621e2e8e67b776a24055b564,2021-05-31,2021-05-30 23:29:30,2021-05-31 09:08:30,2021-05-30,2021-05-31,True,1
8,621e2e8e67b776a24055b564,2021-06-01,2021-05-31 23:42:30,2021-06-01 09:26:30,2021-05-31,2021-06-01,True,1
9,621e2e8e67b776a24055b564,2021-06-02,2021-06-01 23:58:30,2021-06-02 09:32:30,2021-06-01,2021-06-02,True,1


In [6]:
# target calendar_date와 modeling_dataset_daily calendar_date join이 완전히 맞는지 확인
daily_keys = daily[["participant_object_id", "calendar_date"]].copy()
target_keys = target[["participant_object_id", "calendar_date"]].copy()

daily_keys["in_daily"] = True
target_keys["in_target"] = True

merged_keys = target_keys.merge(
    daily_keys,
    on=["participant_object_id", "calendar_date"],
    how="outer",
)

print("target keys:", len(target_keys))
print("daily keys:", len(daily_keys))
print("keys in target not in daily:", merged_keys["in_daily"].isna().sum())
print("keys in daily not in target:", merged_keys["in_target"].isna().sum())

merged_keys[
    merged_keys["in_daily"].isna() | merged_keys["in_target"].isna()
].head(30)

target keys: 3551
daily keys: 3551
keys in target not in daily: 0
keys in daily not in target: 0


,participant_object_id,calendar_date,in_target,in_daily


결과 : target alignment 자체는 일관됨
하지만 same-date feature 사용은 temporal leakage 가능성이 있음

## 2번: feature family별 날짜 의미 / temporal leakage 후보 점검

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
DAILY_DIR = PROJECT_ROOT / "data" / "processed" / "daily_aggregates"

daily_files = sorted(DAILY_DIR.glob("*.csv"))

file_summaries = []
for path in daily_files:
    df = pd.read_csv(path, nrows=5)
    file_summaries.append({
        "file": path.name,
        "columns": list(df.columns),
        "rows_previewed": len(df),
    })

for item in file_summaries:
    print("\n===", item["file"], "===")
    print(item["columns"])


=== fitbit_activity_minutes_daily.csv ===
['participant_object_id', 'calendar_date', 'lightly_active_minutes_sum', 'moderately_active_minutes_sum', 'sedentary_minutes_sum', 'very_active_minutes_sum']

=== fitbit_calories_daily.csv ===
['participant_object_id', 'calendar_date', 'calories_sum', 'calories_record_count']

=== fitbit_daily_hrv_summary_daily.csv ===
['participant_object_id', 'calendar_date', 'hrv_summary_rmssd_mean', 'hrv_summary_nremhr_mean', 'hrv_summary_entropy_mean', 'hrv_summary_record_count']

=== fitbit_daily_spo2_daily.csv ===
['participant_object_id', 'calendar_date', 'spo2_average_value_mean', 'spo2_lower_bound_mean', 'spo2_upper_bound_mean', 'spo2_record_count']

=== fitbit_hrv_details_daily.csv ===
['participant_object_id', 'calendar_date', 'hrv_detail_rmssd_mean', 'hrv_detail_rmssd_std', 'hrv_detail_rmssd_min', 'hrv_detail_rmssd_max', 'hrv_detail_coverage_mean', 'hrv_detail_coverage_std', 'hrv_detail_coverage_min', 'hrv_detail_coverage_max', 'hrv_detail_low_fre

In [8]:
summary_rows = []

for path in daily_files:
    df = pd.read_csv(path)
    
    key_cols = []
    if "participant_object_id" in df.columns:
        key_cols.append("participant_object_id")
    if "calendar_date" in df.columns:
        key_cols.append("calendar_date")
    
    row = {
        "file": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "has_participant": "participant_object_id" in df.columns,
        "has_calendar_date": "calendar_date" in df.columns,
        "duplicate_keys": np.nan,
        "participants": np.nan,
        "min_calendar_date": None,
        "max_calendar_date": None,
    }
    
    if "participant_object_id" in df.columns:
        row["participants"] = df["participant_object_id"].nunique()
    
    if "calendar_date" in df.columns:
        row["min_calendar_date"] = df["calendar_date"].min()
        row["max_calendar_date"] = df["calendar_date"].max()
    
    if key_cols:
        row["duplicate_keys"] = int(df.duplicated(key_cols).sum())
    
    summary_rows.append(row)

daily_file_summary = pd.DataFrame(summary_rows)
daily_file_summary

,file,rows,columns,has_participant,has_calendar_date,duplicate_keys,participants,min_calendar_date,max_calendar_date
0,fitbit_activity_minutes_daily.csv,7083,6,True,True,0,71,2021-05-24,2022-01-22
1,fitbit_calories_daily.csv,6660,4,True,True,0,71,2021-05-24,2022-01-22
2,fitbit_daily_hrv_summary_daily.csv,2475,6,True,True,0,43,2021-05-24,2022-01-22
3,fitbit_daily_spo2_daily.csv,1270,6,True,True,0,28,2021-05-24,2022-01-22
4,fitbit_hrv_details_daily.csv,2583,19,True,True,0,43,2021-05-24,2022-01-22
5,fitbit_respiratory_rate_summary_daily.csv,2495,9,True,True,0,43,2021-05-24,2022-01-21
6,fitbit_resting_heart_rate_daily.csv,12118,5,True,True,0,71,2021-05-24,2022-01-22
7,fitbit_steps_daily.csv,4777,4,True,True,0,71,2021-05-24,2022-01-22
8,fitbit_stress_daily.csv,1876,9,True,True,0,37,2021-05-24,2022-01-22
9,fitbit_wrist_temperature_daily.csv,3304,6,True,True,0,61,2021-05-24,2022-01-22


In [9]:
risk_keywords = [
    "sleep",
    "asleep",
    "awake",
    "wake",
    "rem",
    "deep",
    "light",
    "efficiency",
    "timeinbed",
    "time_in_bed",
    "restless",
    "respiratory",
    "spo2",
    "hrv",
    "temperature",
    "stress",
]

family_risk_rows = []

for path in daily_files:
    df = pd.read_csv(path, nrows=0)
    columns = list(df.columns)
    lower_file = path.name.lower()
    
    risky_cols = []
    for col in columns:
        lower_col = col.lower()
        if any(keyword in lower_col for keyword in risk_keywords):
            risky_cols.append(col)
    
    family_risk_rows.append({
        "file": path.name,
        "columns": len(columns),
        "risk_keyword_columns": len(risky_cols),
        "risk_keyword_column_examples": risky_cols[:20],
        "file_name_has_risk_keyword": any(keyword in lower_file for keyword in risk_keywords),
    })

family_risk_summary = pd.DataFrame(family_risk_rows)
family_risk_summary

,file,columns,risk_keyword_columns,risk_keyword_column_examples,file_name_has_risk_keyword
0,fitbit_activity_minutes_daily.csv,6,1,[lightly_active_minutes_sum],False
1,fitbit_calories_daily.csv,4,0,[],False
2,fitbit_daily_hrv_summary_daily.csv,6,4,"[hrv_summary_rmssd_mean, hrv_summary_nremhr_me...",True
3,fitbit_daily_spo2_daily.csv,6,4,"[spo2_average_value_mean, spo2_lower_bound_mea...",True
4,fitbit_hrv_details_daily.csv,19,17,"[hrv_detail_rmssd_mean, hrv_detail_rmssd_std, ...",True
5,fitbit_respiratory_rate_summary_daily.csv,9,7,"[respiratory_full_sleep_breathing_rate_mean, r...",True
6,fitbit_resting_heart_rate_daily.csv,5,0,[],False
7,fitbit_steps_daily.csv,4,0,[],False
8,fitbit_stress_daily.csv,9,7,"[stress_score_mean, stress_sleep_points_mean, ...",True
9,fitbit_wrist_temperature_daily.csv,6,4,"[wrist_temperature_mean, wrist_temperature_min...",True


In [10]:
encoded_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_encoded.csv"
feature_cols_path = PROJECT_ROOT / "data" / "processed" / "encoded_feature_columns.csv"

encoded = pd.read_csv(encoded_path, nrows=5)
feature_cols = pd.read_csv(feature_cols_path)["feature"].tolist()

leakage_keywords = [
    "sleep",
    "asleep",
    "awake",
    "wake",
    "rem",
    "deep",
    "light",
    "efficiency",
    "timeinbed",
    "time_in_bed",
    "restless",
]

recovery_or_sleep_measurement_keywords = [
    "hrv",
    "spo2",
    "respiratory",
    "wrist_temperature",
    "temperature",
    "stress",
]

direct_sleep_like_features = [
    col for col in feature_cols
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

recovery_like_features = [
    col for col in feature_cols
    if any(keyword in col.lower() for keyword in recovery_or_sleep_measurement_keywords)
]

print("feature count:", len(feature_cols))
print("\ndirect sleep-like features still included:", len(direct_sleep_like_features))
print(direct_sleep_like_features)

print("\nrecovery/sleep-measurement-like features included:", len(recovery_like_features))
print(recovery_like_features[:100])

feature count: 197

direct sleep-like features still included: 14
['hrv_summary_nremhr_mean_missing_ind', 'hrv_summary_nremhr_mean', 'lightly_active_minutes_sum_missing_ind', 'lightly_active_minutes_sum', 'respiratory_full_sleep_breathing_rate_mean_missing_ind', 'respiratory_full_sleep_breathing_rate_mean', 'respiratory_full_sleep_signal_to_noise_mean_missing_ind', 'respiratory_full_sleep_signal_to_noise_mean', 'respiratory_full_sleep_standard_deviation_mean_missing_ind', 'respiratory_full_sleep_standard_deviation_mean', 'respiratory_deep_sleep_breathing_rate_mean_missing_ind', 'respiratory_deep_sleep_breathing_rate_mean', 'respiratory_rem_sleep_breathing_rate_mean_missing_ind', 'respiratory_rem_sleep_breathing_rate_mean']

recovery/sleep-measurement-like features included: 82
['stress_score_mean_missing_ind', 'stress_score_mean', 'stress_responsiveness_points_mean_missing_ind', 'stress_responsiveness_points_mean', 'stress_exertion_points_mean_missing_ind', 'stress_exertion_points_mean

In [11]:
def infer_feature_family(feature: str) -> str:
    f = feature.lower()
    if f.startswith("stress"):
        return "stress"
    if f.startswith("hrv"):
        return "hrv"
    if f.startswith("resting_hr"):
        return "resting_hr"
    if "active_minutes" in f or f.startswith("steps") or f.startswith("calories") or f.startswith("sedentary"):
        return "activity"
    if f.startswith("spo2"):
        return "spo2"
    if f.startswith("respiratory"):
        return "respiratory"
    if "temperature" in f:
        return "temperature"
    if f.startswith("mood") or f.startswith("place") or f.startswith("sema"):
        return "sema"
    if f.startswith("survey"):
        return "survey"
    return "other"

feature_family_summary = (
    pd.DataFrame({"feature": feature_cols})
    .assign(family=lambda x: x["feature"].map(infer_feature_family))
    .groupby("family")
    .agg(
        feature_count=("feature", "size"),
        examples=("feature", lambda s: list(s.head(10))),
    )
    .reset_index()
    .sort_values("feature_count", ascending=False)
)

feature_family_summary

,family,feature_count,examples
4,sema,82,"[sema_response_count_missing_ind, sema_respons..."
1,hrv,42,"[hrv_summary_rmssd_mean_missing_ind, hrv_summa..."
0,activity,16,"[lightly_active_minutes_sum_missing_ind, light..."
7,survey,14,"[survey_response_count_missing_ind, survey_res..."
2,respiratory,12,[respiratory_full_sleep_breathing_rate_mean_mi...
6,stress,12,"[stress_score_mean_missing_ind, stress_score_m..."
5,spo2,8,"[spo2_average_value_mean_missing_ind, spo2_ave..."
8,temperature,8,"[wrist_temperature_mean_missing_ind, wrist_tem..."
3,resting_hr,3,"[resting_hr_resting_heart_rate_mean, resting_h..."


In [12]:
temporal_risk_table = pd.DataFrame([
    {
        "family": "activity / steps / calories",
        "same_date_temporal_risk": "high",
        "reason": "sleep target calendar_date is sleep end date; same-date daytime activity may happen after the sleep event.",
        "recommended_check": "compare same-date features vs previous-day shifted features",
    },
    {
        "family": "stress",
        "same_date_temporal_risk": "medium_to_high",
        "reason": "stress scores may include recovery/sleep-related components or post-sleep same-day signals.",
        "recommended_check": "inspect stress source semantics; compare excluding stress features",
    },
    {
        "family": "HRV",
        "same_date_temporal_risk": "medium_to_high",
        "reason": "Fitbit HRV is often sleep/night-derived; may describe the same sleep episode or recovery state.",
        "recommended_check": "inspect whether HRV date corresponds to sleep night/end date",
    },
    {
        "family": "SpO2 / respiratory",
        "same_date_temporal_risk": "high",
        "reason": "These are often sleep-session measurements and may be too close to the target sleep event.",
        "recommended_check": "run feature-set ablation excluding sleep-session-derived measurements",
    },
    {
        "family": "wrist temperature",
        "same_date_temporal_risk": "medium",
        "reason": "May be nightly/sleep-adjacent depending on Fitbit export semantics.",
        "recommended_check": "inspect source type and date definition",
    },
    {
        "family": "SEMA context/mood",
        "same_date_temporal_risk": "medium",
        "reason": "Same-date survey responses may happen after waking and after the target sleep event.",
        "recommended_check": "shift to previous-day SEMA or exclude same-date responses",
    },
    {
        "family": "participant survey",
        "same_date_temporal_risk": "low",
        "reason": "Mostly participant-level traits or survey counts, but availability/count signals may still matter.",
        "recommended_check": "compare with and without survey/count features",
    },
])

temporal_risk_table

,family,same_date_temporal_risk,reason,recommended_check
0,activity / steps / calories,high,sleep target calendar_date is sleep end date; ...,compare same-date features vs previous-day shi...
1,stress,medium_to_high,stress scores may include recovery/sleep-relat...,inspect stress source semantics; compare exclu...
2,HRV,medium_to_high,Fitbit HRV is often sleep/night-derived; may d...,inspect whether HRV date corresponds to sleep ...
3,SpO2 / respiratory,high,These are often sleep-session measurements and...,run feature-set ablation excluding sleep-sessi...
4,wrist temperature,medium,May be nightly/sleep-adjacent depending on Fit...,inspect source type and date definition
5,SEMA context/mood,medium,Same-date survey responses may happen after wa...,shift to previous-day SEMA or exclude same-dat...
6,participant survey,low,Mostly participant-level traits or survey coun...,compare with and without survey/count features


결론 : 현재 feature에는 직접 sleep outcome은 대부분 제거됐지만,
sleep-adjacent / recovery-derived feature가 상당히 많이 남아 있습니다.

## 3번: feature risk group별 target correlation / missingness / split distribution 진단

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")

encoded_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_encoded.csv"
feature_cols_path = PROJECT_ROOT / "data" / "processed" / "encoded_feature_columns.csv"
dl_split_path = PROJECT_ROOT / "data" / "processed" / "deep_learning_sequences" / "deep_learning_participant_split_assignments.csv"

df = pd.read_csv(encoded_path)
feature_cols = pd.read_csv(feature_cols_path)["feature"].tolist()
dl_splits = pd.read_csv(dl_split_path)

df["participant_object_id"] = df["participant_object_id"].astype(str)
dl_splits["participant_object_id"] = dl_splits["participant_object_id"].astype(str)

split_map = dl_splits.set_index("participant_object_id")["deep_learning_split"].to_dict()
df["deep_learning_split"] = df["participant_object_id"].map(split_map)

print("df shape:", df.shape)
print("feature count:", len(feature_cols))
print(df["deep_learning_split"].value_counts(dropna=False))

df shape: (3551, 201)
feature count: 197
deep_learning_split
train         2425
test           607
validation     519
Name: count, dtype: int64


In [14]:
def assign_risk_group(feature: str) -> str:
    f = feature.lower()

    # Missing indicator 자체는 별도 구분
    if f.endswith("_missing_ind"):
        base = f.replace("_missing_ind", "")
        return assign_risk_group(base) + "__missing_indicator"

    # 명시적으로 sleep session에서 나온 것으로 보이는 feature
    if f.startswith("respiratory"):
        return "high_risk_sleep_session"
    if f.startswith("spo2"):
        return "high_risk_sleep_session"
    if f.startswith("hrv"):
        return "high_risk_sleep_session"
    if "wrist_temperature" in f or "temperature" in f:
        return "high_risk_sleep_session"

    # stress는 recovery/sleep component 가능성이 있어서 중간~높은 위험
    if f.startswith("stress"):
        return "medium_high_risk_stress_recovery"

    # same-date daytime activity는 sleep end date 기준일 때 post-sleep 가능성이 있음
    if (
        f.startswith("steps")
        or f.startswith("calories")
        or "active_minutes" in f
        or f.startswith("sedentary")
    ):
        return "medium_risk_same_date_daytime"

    # resting HR은 daily summary 의미에 따라 중간 위험
    if f.startswith("resting_hr"):
        return "medium_risk_resting_hr"

    # SEMA는 응답 시간이 target sleep 이후일 수 있음
    if f.startswith("sema") or f.startswith("mood") or f.startswith("place"):
        return "medium_risk_same_date_context"

    # survey는 participant-level trait/count 성격
    if f.startswith("survey"):
        return "low_risk_participant_survey"

    return "other"

feature_risk = pd.DataFrame({
    "feature": feature_cols,
    "risk_group": [assign_risk_group(f) for f in feature_cols],
})

feature_risk_summary = (
    feature_risk
    .groupby("risk_group")
    .agg(
        feature_count=("feature", "size"),
        examples=("feature", lambda s: list(s.head(12))),
    )
    .reset_index()
    .sort_values("feature_count", ascending=False)
)

feature_risk_summary

,risk_group,feature_count,examples
8,medium_risk_same_date_context__missing_indicator,41,"[sema_response_count_missing_ind, mood_alert_c..."
7,medium_risk_same_date_context,41,"[sema_response_count, mood_alert_count, mood_a..."
0,high_risk_sleep_session,35,"[hrv_summary_rmssd_mean, hrv_summary_nremhr_me..."
1,high_risk_sleep_session__missing_indicator,35,"[hrv_summary_rmssd_mean_missing_ind, hrv_summa..."
10,medium_risk_same_date_daytime__missing_indicator,8,"[lightly_active_minutes_sum_missing_ind, moder..."
9,medium_risk_same_date_daytime,8,"[lightly_active_minutes_sum, moderately_active..."
2,low_risk_participant_survey,7,"[survey_response_count, survey_bfpt_count, sur..."
3,low_risk_participant_survey__missing_indicator,7,"[survey_response_count_missing_ind, survey_bfp..."
4,medium_high_risk_stress_recovery,6,"[stress_score_mean, stress_responsiveness_poin..."
5,medium_high_risk_stress_recovery__missing_indi...,6,"[stress_score_mean_missing_ind, stress_respons..."


In [15]:
corr_rows = []

for _, row in feature_risk.iterrows():
    feature = row["feature"]
    risk_group = row["risk_group"]

    if df[feature].nunique(dropna=True) <= 1:
        corr = np.nan
    else:
        corr = df[feature].corr(df["good_sleep_label"])

    corr_rows.append({
        "feature": feature,
        "risk_group": risk_group,
        "corr_with_target": corr,
        "abs_corr_with_target": abs(corr) if pd.notna(corr) else np.nan,
        "mean": df[feature].mean(),
        "std": df[feature].std(),
    })

feature_corr = pd.DataFrame(corr_rows)

risk_corr_summary = (
    feature_corr
    .groupby("risk_group")
    .agg(
        features=("feature", "size"),
        mean_abs_corr=("abs_corr_with_target", "mean"),
        max_abs_corr=("abs_corr_with_target", "max"),
        top_feature=("feature", lambda s: None),
    )
    .reset_index()
)

top_by_group = (
    feature_corr
    .sort_values("abs_corr_with_target", ascending=False)
    .groupby("risk_group")
    .head(5)
    .sort_values(["risk_group", "abs_corr_with_target"], ascending=[True, False])
)

risk_corr_summary.sort_values("max_abs_corr", ascending=False)

,risk_group,features,mean_abs_corr,max_abs_corr,top_feature
9,medium_risk_same_date_daytime,8,0.133384,0.288735,None
0,high_risk_sleep_session,35,0.041973,0.221441,None
2,low_risk_participant_survey,7,0.033065,0.125669,None
1,high_risk_sleep_session__missing_indicator,35,0.053248,0.068130,None
7,medium_risk_same_date_context,41,0.022398,0.057414,None
6,medium_risk_resting_hr,3,0.035283,0.049065,None
4,medium_high_risk_stress_recovery,6,0.011379,0.034225,None
8,medium_risk_same_date_context__missing_indicator,41,0.022275,0.022275,None
3,low_risk_participant_survey__missing_indicator,7,0.021381,0.021381,None
10,medium_risk_same_date_daytime__missing_indicator,8,0.017238,0.019129,None


In [16]:
top_by_group[
    [
        "risk_group",
        "feature",
        "corr_with_target",
        "abs_corr_with_target",
        "mean",
        "std",
    ]
]

,risk_group,feature,corr_with_target,abs_corr_with_target,mean,std
53,high_risk_sleep_session,hrv_detail_record_count,0.221441,0.221441,61.557871,43.600626
76,high_risk_sleep_session,respiratory_full_sleep_signal_to_noise_mean,0.082346,0.082346,7.692590,2.469788
100,high_risk_sleep_session,wrist_temperature_record_count,0.073598,0.073598,1225.358209,414.271762
78,high_risk_sleep_session,respiratory_full_sleep_standard_deviation_mean,0.071527,0.071527,1.040472,0.342694
68,high_risk_sleep_session,spo2_lower_bound_mean,-0.066989,0.066989,93.554604,1.029555
71,high_risk_sleep_session__missing_indicator,spo2_record_count_missing_ind,0.068130,0.068130,0.653619,0.475883
67,high_risk_sleep_session__missing_indicator,spo2_lower_bound_mean_missing_ind,0.068130,0.068130,0.653619,0.475883
69,high_risk_sleep_session__missing_indicator,spo2_upper_bound_mean_missing_ind,0.068130,0.068130,0.653619,0.475883
65,high_risk_sleep_session__missing_indicator,spo2_average_value_mean_missing_ind,0.068130,0.068130,0.653619,0.475883
16,high_risk_sleep_session__missing_indicator,hrv_summary_entropy_mean_missing_ind,-0.055436,0.055436,0.306393,0.461060


In [17]:
missing_indicator_features = [
    f for f in feature_cols
    if f.endswith("_missing_ind")
]

missing_corr_rows = []

for feature in missing_indicator_features:
    corr = df[feature].corr(df["good_sleep_label"]) if df[feature].nunique() > 1 else np.nan
    missing_corr_rows.append({
        "feature": feature,
        "risk_group": assign_risk_group(feature),
        "missing_rate": df[feature].mean(),
        "corr_with_target": corr,
        "abs_corr_with_target": abs(corr) if pd.notna(corr) else np.nan,
    })

missing_corr = pd.DataFrame(missing_corr_rows)

missing_corr.sort_values("abs_corr_with_target", ascending=False).head(30)

,feature,risk_group,missing_rate,corr_with_target,abs_corr_with_target
31,spo2_average_value_mean_missing_ind,high_risk_sleep_session__missing_indicator,0.653619,0.068130,0.068130
32,spo2_lower_bound_mean_missing_ind,high_risk_sleep_session__missing_indicator,0.653619,0.068130,0.068130
33,spo2_upper_bound_mean_missing_ind,high_risk_sleep_session__missing_indicator,0.653619,0.068130,0.068130
34,spo2_record_count_missing_ind,high_risk_sleep_session__missing_indicator,0.653619,0.068130,0.068130
6,hrv_summary_rmssd_mean_missing_ind,high_risk_sleep_session__missing_indicator,0.306393,-0.055436,0.055436
9,hrv_summary_record_count_missing_ind,high_risk_sleep_session__missing_indicator,0.306393,-0.055436,0.055436
8,hrv_summary_entropy_mean_missing_ind,high_risk_sleep_session__missing_indicator,0.306393,-0.055436,0.055436
7,hrv_summary_nremhr_mean_missing_ind,high_risk_sleep_session__missing_indicator,0.306393,-0.055436,0.055436
23,hrv_detail_high_frequency_std_missing_ind,high_risk_sleep_session__missing_indicator,0.300479,-0.052891,0.052891
19,hrv_detail_low_frequency_std_missing_ind,high_risk_sleep_session__missing_indicator,0.300479,-0.052891,0.052891


In [18]:
split_group_rows = []

for risk_group, group_features in feature_risk.groupby("risk_group"):
    cols = group_features["feature"].tolist()
    
    for split_name, split_df in df.groupby("deep_learning_split"):
        split_group_rows.append({
            "risk_group": risk_group,
            "split": split_name,
            "rows": len(split_df),
            "participants": split_df["participant_object_id"].nunique(),
            "target_mean": split_df["good_sleep_label"].mean(),
            "group_feature_mean_mean": split_df[cols].mean().mean(),
            "group_feature_abs_mean_mean": split_df[cols].abs().mean().mean(),
            "group_feature_std_mean": split_df[cols].std().mean(),
        })

split_group_summary = pd.DataFrame(split_group_rows)

split_group_summary.sort_values(["risk_group", "split"])

,risk_group,split,rows,participants,target_mean,group_feature_mean_mean,group_feature_abs_mean_mean,group_feature_std_mean
0,high_risk_sleep_session,test,607,14,0.369028,348.725105,349.219978,244.418913
1,high_risk_sleep_session,train,2425,44,0.400000,410.307331,410.805046,307.065118
2,high_risk_sleep_session,validation,519,11,0.393064,375.448980,375.988336,216.961254
3,high_risk_sleep_session__missing_indicator,test,607,14,0.369028,0.439162,0.439162,0.475961
4,high_risk_sleep_session__missing_indicator,train,2425,44,0.400000,0.267935,0.267935,0.413620
5,high_risk_sleep_session__missing_indicator,validation,519,11,0.393064,0.391577,0.391577,0.459084
6,low_risk_participant_survey,test,607,14,0.369028,3.819722,3.819722,2.345518
7,low_risk_participant_survey,train,2425,44,0.400000,4.276524,4.276524,2.080478
8,low_risk_participant_survey,validation,519,11,0.393064,3.606386,3.606386,2.457068
9,low_risk_participant_survey__missing_indicator,test,607,14,0.369028,0.108731,0.108731,0.311559


In [19]:
missing_split_rows = []

for risk_group, group_features in feature_risk[feature_risk["feature"].str.endswith("_missing_ind")].groupby("risk_group"):
    cols = group_features["feature"].tolist()
    
    for split_name, split_df in df.groupby("deep_learning_split"):
        missing_split_rows.append({
            "risk_group": risk_group,
            "split": split_name,
            "rows": len(split_df),
            "participants": split_df["participant_object_id"].nunique(),
            "target_mean": split_df["good_sleep_label"].mean(),
            "avg_missing_indicator_rate": split_df[cols].mean().mean(),
            "max_missing_indicator_rate": split_df[cols].mean().max(),
        })

missing_split_summary = pd.DataFrame(missing_split_rows)

missing_split_summary.sort_values(["risk_group", "split"])

,risk_group,split,rows,participants,target_mean,avg_missing_indicator_rate,max_missing_indicator_rate
0,high_risk_sleep_session__missing_indicator,test,607,14,0.369028,0.439162,0.738056
1,high_risk_sleep_session__missing_indicator,train,2425,44,0.400000,0.267935,0.625567
2,high_risk_sleep_session__missing_indicator,validation,519,11,0.393064,0.391577,0.685934
3,low_risk_participant_survey__missing_indicator,test,607,14,0.369028,0.108731,0.108731
4,low_risk_participant_survey__missing_indicator,train,2425,44,0.400000,0.001649,0.001649
5,low_risk_participant_survey__missing_indicator,validation,519,11,0.393064,0.019268,0.019268
6,medium_high_risk_stress_recovery__missing_indi...,test,607,14,0.369028,0.457990,0.457990
7,medium_high_risk_stress_recovery__missing_indi...,train,2425,44,0.400000,0.454845,0.454845
8,medium_high_risk_stress_recovery__missing_indi...,validation,519,11,0.393064,0.587669,0.587669
9,medium_risk_same_date_context__missing_indicator,test,607,14,0.369028,0.390445,0.390445


In [20]:
participant_rows = []

for participant_id, g in df.groupby("participant_object_id"):
    row = {
        "participant_object_id": participant_id,
        "deep_learning_split": g["deep_learning_split"].iloc[0],
        "rows": len(g),
        "target_mean": g["good_sleep_label"].mean(),
    }

    for risk_group, group_features in feature_risk.groupby("risk_group"):
        missing_cols = [
            f for f in group_features["feature"].tolist()
            if f.endswith("_missing_ind")
        ]
        if missing_cols:
            row[f"{risk_group}_avg_missing_rate"] = g[missing_cols].mean().mean()

    participant_rows.append(row)

participant_risk = pd.DataFrame(participant_rows)

participant_risk_summary = (
    participant_risk
    .groupby("deep_learning_split")
    .agg(
        participants=("participant_object_id", "nunique"),
        rows_mean=("rows", "mean"),
        rows_min=("rows", "min"),
        rows_max=("rows", "max"),
        target_mean=("target_mean", "mean"),
    )
    .reset_index()
)

display(participant_risk_summary)
display(participant_risk.sort_values(["deep_learning_split", "target_mean"]).head(30))
display(participant_risk.sort_values(["deep_learning_split", "target_mean"]).tail(30))

,deep_learning_split,participants,rows_mean,rows_min,rows_max,target_mean
0,test,14,43.357143,2,83,0.323500
1,train,44,55.113636,1,220,0.375865
2,validation,11,47.181818,1,93,0.360404


,participant_object_id,deep_learning_split,rows,target_mean,high_risk_sleep_session__missing_indicator_avg_missing_rate,low_risk_participant_survey__missing_indicator_avg_missing_rate,medium_high_risk_stress_recovery__missing_indicator_avg_missing_rate,medium_risk_same_date_context__missing_indicator_avg_missing_rate,medium_risk_same_date_daytime__missing_indicator_avg_missing_rate
63,621e36bb67b776a240b40d64,test,2,0.000000,1.000000,0.0,1.000000,0.000000,0.000000
65,621e36dd67b776a240ce9a45,test,3,0.000000,1.000000,0.0,0.000000,1.000000,0.000000
29,621e324e67b776a2400191cb,test,68,0.102941,0.900840,0.0,1.000000,0.088235,0.000000
64,621e36c267b776a240ba2756,test,44,0.159091,0.890909,0.0,0.000000,0.000000,0.000000
18,621e309b67b776a240b532b0,test,54,0.166667,0.894180,0.0,1.000000,1.000000,0.000000
33,621e32af67b776a24045b4cf,test,58,0.172414,0.022167,0.0,0.000000,0.000000,0.017241
17,621e309267b776a240ae1cdb,test,17,0.235294,0.899160,0.0,1.000000,0.294118,0.000000
8,621e2f6167b776a240e082a9,test,45,0.288889,0.112381,0.0,0.000000,0.044444,0.000000
4,621e2efa67b776a2409dd1c3,test,66,0.469697,0.778355,1.0,1.000000,1.000000,0.000000
55,621e34f767b776a240de4e1a,test,2,0.500000,1.000000,0.0,1.000000,0.000000,0.000000


,participant_object_id,deep_learning_split,rows,target_mean,high_risk_sleep_session__missing_indicator_avg_missing_rate,low_risk_participant_survey__missing_indicator_avg_missing_rate,medium_high_risk_stress_recovery__missing_indicator_avg_missing_rate,medium_risk_same_date_context__missing_indicator_avg_missing_rate,medium_risk_same_date_daytime__missing_indicator_avg_missing_rate
23,621e310d67b776a24003096d,train,67,0.388060,0.061834,0.0,1.000000,0.462687,0.000000
68,621e375b67b776a240290cdc,train,66,0.409091,0.177922,0.0,0.000000,0.045455,0.000000
46,621e33ed67b776a2401cf5f7,train,62,0.419355,0.889401,0.0,1.000000,0.161290,0.000000
16,621e301e67b776a240608a72,train,53,0.452830,0.890027,0.0,0.000000,0.037736,0.000000
28,621e323667b776a240f19134,train,60,0.466667,0.122857,0.0,0.000000,0.200000,0.000000
12,621e2fce67b776a240279baa,train,64,0.484375,0.016518,0.0,0.000000,0.015625,0.015625
7,621e2f5767b776a240d8f9d6,train,22,0.500000,0.901299,0.0,0.000000,0.045455,0.000000
21,621e30e467b776a240e817c7,train,30,0.500000,0.144762,0.0,0.000000,0.000000,0.000000
2,621e2ed667b776a24085d8d1,train,76,0.526316,0.221053,0.0,1.000000,0.289474,0.000000
66,621e36f967b776a240e5e7c9,train,45,0.600000,0.043175,0.0,0.000000,0.000000,0.000000


현재까지의 데이터 진단 결론
1. target 날짜는 sleep end date 기준
2. same-date activity는 sleep 이후 정보일 수 있음
3. high-risk sleep-session/recovery feature가 많음
4. split 간 missingness/coverage 분포 차이가 큼
5. test participants는 participant-level target mean이 낮음

In [25]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_PATH = PROCESSED_DIR / "modeling_dataset_encoded.csv"
FEATURE_PATH = PROCESSED_DIR / "encoded_feature_columns.csv"
SPLIT_PATH = PROCESSED_DIR / "deep_learning_sequences" / "deep_learning_participant_split_assignments.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
feature_df = pd.read_csv(FEATURE_PATH, encoding="utf-8-sig")
split_df = pd.read_csv(SPLIT_PATH, encoding="utf-8-sig")

feature_cols = feature_df["feature"].tolist()

split_df = split_df[["participant_object_id", "deep_learning_split"]].drop_duplicates()
df = df.merge(split_df, on="participant_object_id", how="left")

print("df shape:", df.shape)
print("feature count:", len(feature_cols))
print(df["deep_learning_split"].value_counts(dropna=False))


def assign_risk_group(feature_name):
    base = feature_name.replace("_missing_ind", "")
    is_missing = feature_name.endswith("_missing_ind")

    high_sleep_keywords = [
        "hrv_summary",
        "hrv_detail",
        "spo2",
        "respiratory",
        "wrist_temperature",
    ]
    stress_keywords = ["stress"]
    same_date_context_keywords = [
        "sema",
        "mood",
        "place",
    ]
    same_date_daytime_keywords = [
        "active_minutes",
        "steps",
        "calories",
        "sedentary",
    ]
    survey_keywords = ["survey"]
    resting_hr_keywords = ["resting_hr"]

    if any(k in base for k in high_sleep_keywords):
        group = "high_risk_sleep_session"
    elif any(k in base for k in stress_keywords):
        group = "medium_high_risk_stress_recovery"
    elif any(k in base for k in same_date_context_keywords):
        group = "medium_risk_same_date_context"
    elif any(k in base for k in same_date_daytime_keywords):
        group = "medium_risk_same_date_daytime"
    elif any(k in base for k in survey_keywords):
        group = "low_risk_participant_survey"
    elif any(k in base for k in resting_hr_keywords):
        group = "medium_risk_resting_hr"
    else:
        group = "other"

    if is_missing:
        group = f"{group}__missing_indicator"

    return group


feature_risk = pd.DataFrame({
    "feature": feature_cols,
    "risk_group": [assign_risk_group(c) for c in feature_cols],
    "is_missing_indicator": [c.endswith("_missing_ind") for c in feature_cols],
})


def compute_shift(data, feature_columns, compare_split, reference_split="train"):
    ref = data[data["deep_learning_split"] == reference_split]
    cmp = data[data["deep_learning_split"] == compare_split]

    rows = []

    for col in feature_columns:
        ref_values = pd.to_numeric(ref[col], errors="coerce")
        cmp_values = pd.to_numeric(cmp[col], errors="coerce")

        ref_mean = ref_values.mean()
        cmp_mean = cmp_values.mean()
        ref_std = ref_values.std(ddof=0)
        cmp_std = cmp_values.std(ddof=0)

        pooled_std = np.sqrt((ref_std ** 2 + cmp_std ** 2) / 2)

        if pooled_std == 0 or pd.isna(pooled_std):
            smd = 0.0
        else:
            smd = (cmp_mean - ref_mean) / pooled_std

        rows.append({
            "feature": col,
            "reference_split": reference_split,
            "compare_split": compare_split,
            "train_mean": ref_mean,
            f"{compare_split}_mean": cmp_mean,
            "train_std": ref_std,
            f"{compare_split}_std": cmp_std,
            "smd": smd,
            "abs_smd": abs(smd),
            "mean_diff": cmp_mean - ref_mean,
        })

    result = pd.DataFrame(rows)
    result = result.merge(feature_risk, on="feature", how="left")
    return result.sort_values("abs_smd", ascending=False)


validation_shift = compute_shift(df, feature_cols, "validation")
test_shift = compute_shift(df, feature_cols, "test")

all_shift = pd.concat([validation_shift, test_shift], ignore_index=True)

shift_summary = (
    all_shift
    .groupby(["compare_split", "risk_group"], as_index=False)
    .agg(
        feature_count=("feature", "count"),
        mean_abs_smd=("abs_smd", "mean"),
        median_abs_smd=("abs_smd", "median"),
        max_abs_smd=("abs_smd", "max"),
        features_abs_smd_ge_0_10=("abs_smd", lambda s: int((s >= 0.10).sum())),
        features_abs_smd_ge_0_20=("abs_smd", lambda s: int((s >= 0.20).sum())),
        features_abs_smd_ge_0_50=("abs_smd", lambda s: int((s >= 0.50).sum())),
    )
    .sort_values(["compare_split", "mean_abs_smd"], ascending=[True, False])
)

missing_shift = all_shift[all_shift["is_missing_indicator"]].copy()
missing_shift["abs_rate_diff"] = missing_shift["mean_diff"].abs()

target_summary = (
    df.groupby("deep_learning_split")
    .agg(
        rows=("good_sleep_label", "size"),
        participants=("participant_object_id", "nunique"),
        target_mean=("good_sleep_label", "mean"),
        min_date=("calendar_date", "min"),
        max_date=("calendar_date", "max"),
    )
    .reset_index()
)

participant_target_summary = (
    df.groupby(["deep_learning_split", "participant_object_id"])
    .agg(
        rows=("good_sleep_label", "size"),
        target_mean=("good_sleep_label", "mean"),
    )
    .reset_index()
    .groupby("deep_learning_split")
    .agg(
        participants=("participant_object_id", "nunique"),
        rows_mean=("rows", "mean"),
        rows_min=("rows", "min"),
        rows_max=("rows", "max"),
        participant_target_mean=("target_mean", "mean"),
        participant_target_std=("target_mean", "std"),
    )
    .reset_index()
)

print("\n=== Risk group shift summary ===")
display(shift_summary)

print("\n=== Train vs validation: top shifted features ===")
display(validation_shift.head(30))

print("\n=== Train vs test: top shifted features ===")
display(test_shift.head(30))

print("\n=== Missing indicator shift: top features ===")
display(
    missing_shift
    .sort_values(["compare_split", "abs_rate_diff"], ascending=[True, False])
    .head(40)
)

print("\n=== Split target summary ===")
display(target_summary)

print("\n=== Participant-level target summary ===")
display(participant_target_summary)

df shape: (3551, 201)
feature count: 197
deep_learning_split
train         2425
test           607
validation     519
Name: count, dtype: int64

=== Risk group shift summary ===


,compare_split,risk_group,feature_count,mean_abs_smd,median_abs_smd,max_abs_smd,features_abs_smd_ge_0_10,features_abs_smd_ge_0_20,features_abs_smd_ge_0_50
3,test,low_risk_participant_survey__missing_indicator,7,0.482381,0.482381,0.482381,7,7,0
1,test,high_risk_sleep_session__missing_indicator,35,0.383121,0.404941,0.409860,35,35,0
2,test,low_risk_participant_survey,7,0.276387,0.213014,0.487281,6,4,0
0,test,high_risk_sleep_session,35,0.219710,0.223552,0.541385,32,18,1
9,test,medium_risk_same_date_daytime,8,0.193509,0.129815,0.451421,4,3,0
8,test,medium_risk_same_date_context__missing_indicator,41,0.157385,0.157385,0.157385,41,0,0
6,test,medium_risk_resting_hr,3,0.118987,0.110328,0.211395,2,1,0
7,test,medium_risk_same_date_context,41,0.109766,0.094905,0.312138,19,8,0
10,test,medium_risk_same_date_daytime__missing_indicator,8,0.050821,0.038512,0.087749,0,0,0
4,test,medium_high_risk_stress_recovery,6,0.027306,0.020060,0.068140,0,0,0



=== Train vs validation: top shifted features ===


,feature,reference_split,compare_split,train_mean,validation_mean,train_std,validation_std,smd,abs_smd,mean_diff,risk_group,is_missing_indicator
196,survey_ttmspbf_count,train,validation,1.802887,1.202312,0.952431,0.641766,-0.739539,0.739539,-0.600574,low_risk_participant_survey,False
188,survey_breq_count,train,validation,1.674227,1.073218,0.946538,0.759294,-0.700445,0.700445,-0.601009,low_risk_participant_survey,False
15,hrv_summary_nremhr_mean,train,validation,63.716172,59.302486,9.094372,11.228755,-0.431975,0.431975,-4.413686,high_risk_sleep_session,False
80,respiratory_deep_sleep_breathing_rate_mean,train,validation,13.612014,15.296339,4.577016,4.030587,0.390571,0.390571,1.684325,high_risk_sleep_session,False
54,resting_hr_resting_heart_rate_mean,train,validation,65.999091,61.548914,10.505735,12.606338,-0.383515,0.383515,-4.450177,medium_risk_resting_hr,False
96,wrist_temperature_min,train,validation,-6.817087,-7.463095,1.963356,1.644519,-0.356720,0.356720,-0.646008,high_risk_sleep_session,False
98,wrist_temperature_max,train,validation,2.141029,1.810393,1.040084,0.907080,-0.338818,0.338818,-0.330636,high_risk_sleep_session,False
79,respiratory_deep_sleep_breathing_rate_mean_mis...,train,validation,0.245361,0.396917,0.430301,0.489259,0.328953,0.328953,0.151556,high_risk_sleep_session__missing_indicator,True
81,respiratory_rem_sleep_breathing_rate_mean_miss...,train,validation,0.245361,0.396917,0.430301,0.489259,0.328953,0.328953,0.151556,high_risk_sleep_session__missing_indicator,True
75,respiratory_full_sleep_signal_to_noise_mean_mi...,train,validation,0.245361,0.396917,0.430301,0.489259,0.328953,0.328953,0.151556,high_risk_sleep_session__missing_indicator,True



=== Train vs test: top shifted features ===


...

,feature,reference_split,compare_split,train_mean,validation_mean,train_std,validation_std,smd,abs_smd,mean_diff,risk_group,is_missing_indicator,test_mean,test_std,abs_rate_diff
209,hrv_detail_coverage_std_missing_ind,train,test,0.246186,NaN,0.430788,NaN,0.409860,0.409860,0.190388,high_risk_sleep_session__missing_indicator,True,0.436573,0.495961,0.190388
210,hrv_detail_high_frequency_std_missing_ind,train,test,0.246186,NaN,0.430788,NaN,0.409860,0.409860,0.190388,high_risk_sleep_session__missing_indicator,True,0.436573,0.495961,0.190388
211,hrv_detail_low_frequency_std_missing_ind,train,test,0.246186,NaN,0.430788,NaN,0.409860,0.409860,0.190388,high_risk_sleep_session__missing_indicator,True,0.436573,0.495961,0.190388
212,hrv_detail_rmssd_std_missing_ind,train,test,0.246186,NaN,0.430788,NaN,0.409860,0.409860,0.190388,high_risk_sleep_session__missing_indicator,True,0.436573,0.495961,0.190388
213,hrv_detail_low_frequency_min_missing_ind,train,test,0.245361,NaN,0.430301,NaN,0.408388,0.408388,0.189565,high_risk_sleep_session__missing_indicator,True,0.434926,0.495747,0.189565
214,hrv_detail_rmssd_min_missing_ind,train,test,0.245361,NaN,0.430301,NaN,0.408388,0.408388,0.189565,high_risk_sleep_session__missing_indicator,True,0.434926,0.495747,0.189565
215,hrv_detail_record_count_missing_ind,train,test,0.245361,NaN,0.430301,NaN,0.408388,0.408388,0.189565,high_risk_sleep_session__missing_indicator,True,0.434926,0.495747,0.189565
216,hrv_detail_low_frequency_max_missing_ind,train,test,0.245361,NaN,0.430301,NaN,0.408388,0.408388,0.189565,high_risk_sleep_session__missing_indicator,True,0.434926,0.495747,0.189565
217,hrv_detail_high_frequency_min_missing_ind,train,test,0.245361,NaN,0.430301,NaN,0.408388,0.408388,0.189565,high_risk_sleep_session__missing_indicator,True,0.434926,0.495747,0.189565
218,hrv_detail_rmssd_max_missing_ind,train,test,0.245361,NaN,0.430301,NaN,0.408388,0.408388,0.189565,high_risk_sleep_session__missing_indicator,True,0.434926,0.495747,0.189565



=== Split target summary ===


,deep_learning_split,rows,participants,target_mean,min_date,max_date
0,test,607,14,0.369028,2021-05-24,2022-01-22
1,train,2425,44,0.400000,2021-05-24,2022-01-21
2,validation,519,11,0.393064,2021-05-24,2022-01-21



=== Participant-level target summary ===


,deep_learning_split,participants,rows_mean,rows_min,rows_max,participant_target_mean,participant_target_std
0,test,14,43.357143,2,83,0.323500,0.240633
1,train,44,55.113636,1,220,0.375865,0.250274
2,validation,11,47.181818,1,93,0.360404,0.233278


In [26]:
# 5번. Ablation 실험용 feature subset 설계
# 목적:
# - 현재 full feature set에서 어떤 위험군을 제거할 수 있는지 확인
# - 이후 deep learning sequence dataset을 feature subset별로 다시 만들기 위한 준비
# - 여기서는 파일 저장/학습은 하지 않음

feature_groups = feature_risk.copy()

full_features = feature_groups["feature"].tolist()

no_high_sleep_features = feature_groups.loc[
    ~feature_groups["risk_group"].str.startswith("high_risk_sleep_session"),
    "feature"
].tolist()

no_high_sleep_no_stress_features = feature_groups.loc[
    ~feature_groups["risk_group"].str.startswith("high_risk_sleep_session")
    & ~feature_groups["risk_group"].str.startswith("medium_high_risk_stress_recovery"),
    "feature"
].tolist()

daytime_context_survey_features = feature_groups.loc[
    feature_groups["risk_group"].str.startswith("medium_risk_same_date_daytime")
    | feature_groups["risk_group"].str.startswith("medium_risk_same_date_context")
    | feature_groups["risk_group"].str.startswith("low_risk_participant_survey")
    | feature_groups["risk_group"].str.startswith("medium_risk_resting_hr"),
    "feature"
].tolist()

daytime_only_features = feature_groups.loc[
    feature_groups["risk_group"].str.startswith("medium_risk_same_date_daytime")
    | feature_groups["risk_group"].str.startswith("medium_risk_resting_hr"),
    "feature"
].tolist()

subset_summary = pd.DataFrame([
    {
        "subset_name": "full_current",
        "feature_count": len(full_features),
        "description": "현재 전체 feature set",
    },
    {
        "subset_name": "no_high_sleep_session",
        "feature_count": len(no_high_sleep_features),
        "description": "HRV, SpO2, respiratory, wrist temperature 및 해당 missing indicator 제거",
    },
    {
        "subset_name": "no_high_sleep_session_no_stress",
        "feature_count": len(no_high_sleep_no_stress_features),
        "description": "sleep-session 계열과 stress/recovery 계열 제거",
    },
    {
        "subset_name": "daytime_context_survey",
        "feature_count": len(daytime_context_survey_features),
        "description": "daytime activity + SEMA/context + participant survey + resting HR",
    },
    {
        "subset_name": "daytime_only",
        "feature_count": len(daytime_only_features),
        "description": "daytime activity + resting HR만 사용",
    },
])

display(subset_summary)

for subset_name, features in {
    "full_current": full_features,
    "no_high_sleep_session": no_high_sleep_features,
    "no_high_sleep_session_no_stress": no_high_sleep_no_stress_features,
    "daytime_context_survey": daytime_context_survey_features,
    "daytime_only": daytime_only_features,
}.items():
    print("\n" + "=" * 80)
    print(subset_name, "feature count:", len(features))
    print(features[:50])

,subset_name,feature_count,description
0,full_current,197,현재 전체 feature set
1,no_high_sleep_session,127,"HRV, SpO2, respiratory, wrist temperature 및 해당..."
2,no_high_sleep_session_no_stress,115,sleep-session 계열과 stress/recovery 계열 제거
3,daytime_context_survey,115,daytime activity + SEMA/context + participant ...
4,daytime_only,19,daytime activity + resting HR만 사용



full_current feature count: 197
['stress_score_mean_missing_ind', 'stress_score_mean', 'stress_responsiveness_points_mean_missing_ind', 'stress_responsiveness_points_mean', 'stress_exertion_points_mean_missing_ind', 'stress_exertion_points_mean', 'stress_ready_rate_missing_ind', 'stress_ready_rate', 'stress_calculation_failed_rate_missing_ind', 'stress_calculation_failed_rate', 'stress_record_count_missing_ind', 'stress_record_count', 'hrv_summary_rmssd_mean_missing_ind', 'hrv_summary_rmssd_mean', 'hrv_summary_nremhr_mean_missing_ind', 'hrv_summary_nremhr_mean', 'hrv_summary_entropy_mean_missing_ind', 'hrv_summary_entropy_mean', 'hrv_summary_record_count_missing_ind', 'hrv_summary_record_count', 'hrv_detail_rmssd_mean_missing_ind', 'hrv_detail_rmssd_mean', 'hrv_detail_rmssd_std_missing_ind', 'hrv_detail_rmssd_std', 'hrv_detail_rmssd_min_missing_ind', 'hrv_detail_rmssd_min', 'hrv_detail_rmssd_max_missing_ind', 'hrv_detail_rmssd_max', 'hrv_detail_coverage_mean_missing_ind', 'hrv_detail_

In [27]:
# 6번. Ablation feature subset 목록 저장
# 목적:
# - 각 feature subset을 CSV로 저장
# - 이후 sequence dataset 생성 스크립트에서 subset별 tensor를 만들 수 있게 준비
# - 저장 위치: data/processed/deep_learning_feature_subsets/

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
SUBSET_DIR = PROJECT_ROOT / "data" / "processed" / "deep_learning_feature_subsets"
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

feature_subsets = {
    "full_current": full_features,
    "no_high_sleep_session": no_high_sleep_features,
    "no_high_sleep_session_no_stress": no_high_sleep_no_stress_features,
    "daytime_context_survey": daytime_context_survey_features,
    "daytime_only": daytime_only_features,
}

saved_rows = []

for subset_name, features in feature_subsets.items():
    out_path = SUBSET_DIR / f"{subset_name}_features.csv"

    out_df = pd.DataFrame({
        "feature": features,
        "risk_group": [
            feature_risk.loc[feature_risk["feature"] == feature, "risk_group"].iloc[0]
            for feature in features
        ],
    })

    out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    saved_rows.append({
        "subset_name": subset_name,
        "feature_count": len(features),
        "path": str(out_path),
    })

saved_summary = pd.DataFrame(saved_rows)

display(saved_summary)

,subset_name,feature_count,path
0,full_current,197,c:\workSpace\DeepLearnin_sleep\data\processed\...
1,no_high_sleep_session,127,c:\workSpace\DeepLearnin_sleep\data\processed\...
2,no_high_sleep_session_no_stress,115,c:\workSpace\DeepLearnin_sleep\data\processed\...
3,daytime_context_survey,115,c:\workSpace\DeepLearnin_sleep\data\processed\...
4,daytime_only,19,c:\workSpace\DeepLearnin_sleep\data\processed\...


In [28]:
# 8번. Feature subset sequence tensor 검증
# 목적:
# - subset별/window별/train-validation-test tensor가 정상 생성되었는지 확인
# - feature count, sample count, y 분포, sample_index row 수가 서로 맞는지 확인
# - 학습 전에 파일 구조 문제를 먼저 잡기 위한 검증 셀

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
SUBSET_SEQUENCE_ROOT = PROJECT_ROOT / "data" / "processed" / "deep_learning_sequences_by_subset"
SUMMARY_PATH = SUBSET_SEQUENCE_ROOT / "sequence_tensor_summary_by_subset.csv"

summary = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")

print("SUBSET_SEQUENCE_ROOT:", SUBSET_SEQUENCE_ROOT)
print("summary shape:", summary.shape)
display(summary)

validation_rows = []

for _, row in summary.iterrows():
    subset_name = row["subset_name"]
    window = int(row["window"])
    split = row["split"]

    subset_root = SUBSET_SEQUENCE_ROOT / subset_name
    window_root = subset_root / f"window_{window}"
    npz_path = window_root / f"{split}.npz"
    feature_path = subset_root / "feature_columns.csv"
    sample_index_path = window_root / "sample_index.csv"

    feature_df = pd.read_csv(feature_path, encoding="utf-8-sig")
    sample_index = pd.read_csv(sample_index_path, encoding="utf-8-sig")
    arrays = np.load(npz_path, allow_pickle=True)

    X_sequence = arrays["X_sequence"]
    X_mlp_flattened = arrays["X_mlp_flattened"]
    X_mlp_current_day = arrays["X_mlp_current_day"]
    y = arrays["y"]
    feature_names = arrays["feature_names"]

    sample_index_split = sample_index[sample_index["split"] == split]

    validation_rows.append({
        "subset_name": subset_name,
        "window": window,
        "split": split,
        "npz_exists": npz_path.exists(),
        "feature_csv_count": len(feature_df),
        "feature_npz_count": len(feature_names),
        "summary_features": int(row["features"]),
        "samples_npz": len(y),
        "samples_summary": int(row["samples"]),
        "samples_index": len(sample_index_split),
        "X_sequence_shape": str(X_sequence.shape),
        "X_mlp_flattened_shape": str(X_mlp_flattened.shape),
        "X_mlp_current_day_shape": str(X_mlp_current_day.shape),
        "y_mean": float(y.mean()) if len(y) else np.nan,
        "has_nan_X_sequence": bool(np.isnan(X_sequence).any()),
        "has_inf_X_sequence": bool(np.isinf(X_sequence).any()),
        "feature_count_match": len(feature_df) == len(feature_names) == int(row["features"]),
        "sample_count_match": len(y) == int(row["samples"]) == len(sample_index_split),
        "sequence_shape_match": X_sequence.shape == (len(y), window, len(feature_names)),
        "mlp_flattened_shape_match": X_mlp_flattened.shape == (len(y), window * len(feature_names)),
        "mlp_current_day_shape_match": X_mlp_current_day.shape == (len(y), len(feature_names)),
    })

validation_df = pd.DataFrame(validation_rows)

display(validation_df)

problem_cols = [
    "npz_exists",
    "feature_count_match",
    "sample_count_match",
    "sequence_shape_match",
    "mlp_flattened_shape_match",
    "mlp_current_day_shape_match",
]

problems = validation_df[
    (validation_df[problem_cols] == False).any(axis=1)
    | validation_df["has_nan_X_sequence"]
    | validation_df["has_inf_X_sequence"]
]

print("problem rows:", len(problems))
display(problems)

compact_summary = (
    validation_df
    .groupby(["subset_name", "window"], as_index=False)
    .agg(
        feature_count=("feature_npz_count", "first"),
        train_samples=("samples_npz", lambda s: int(s.iloc[0])),
        any_nan=("has_nan_X_sequence", "any"),
        any_inf=("has_inf_X_sequence", "any"),
        all_feature_count_match=("feature_count_match", "all"),
        all_sample_count_match=("sample_count_match", "all"),
        all_shape_match=("sequence_shape_match", "all"),
    )
)

display(compact_summary)

SUBSET_SEQUENCE_ROOT: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_sequences_by_subset
summary shape: (45, 13)


,subset_name,window,split,samples,participants,features,sequence_shape,mlp_flattened_shape,mlp_current_day_shape,class_0_samples,class_1_samples,good_sleep_label_mean,tensor_path
0,daytime_context_survey,7,train,1393,35,115,1393 x 7 x 115,1393 x 805,1393 x 115,831,562,0.403446,c:\workSpace\DeepLearnin_sleep\data\processed\...
1,daytime_context_survey,7,validation,269,9,115,269 x 7 x 115,269 x 805,269 x 115,163,106,0.394052,c:\workSpace\DeepLearnin_sleep\data\processed\...
2,daytime_context_survey,7,test,314,9,115,314 x 7 x 115,314 x 805,314 x 115,182,132,0.420382,c:\workSpace\DeepLearnin_sleep\data\processed\...
3,daytime_context_survey,14,train,933,27,115,933 x 14 x 115,933 x 1610,933 x 115,576,357,0.382637,c:\workSpace\DeepLearnin_sleep\data\processed\...
4,daytime_context_survey,14,validation,146,7,115,146 x 14 x 115,146 x 1610,146 x 115,95,51,0.349315,c:\workSpace\DeepLearnin_sleep\data\processed\...
5,daytime_context_survey,14,test,214,5,115,214 x 14 x 115,214 x 1610,214 x 115,116,98,0.457944,c:\workSpace\DeepLearnin_sleep\data\processed\...
6,daytime_context_survey,30,train,435,19,115,435 x 30 x 115,435 x 3450,435 x 115,287,148,0.340230,c:\workSpace\DeepLearnin_sleep\data\processed\...
7,daytime_context_survey,30,validation,55,3,115,55 x 30 x 115,55 x 3450,55 x 115,42,13,0.236364,c:\workSpace\DeepLearnin_sleep\data\processed\...
8,daytime_context_survey,30,test,106,4,115,106 x 30 x 115,106 x 3450,106 x 115,43,63,0.594340,c:\workSpace\DeepLearnin_sleep\data\processed\...
9,daytime_only,7,train,1393,35,19,1393 x 7 x 19,1393 x 133,1393 x 19,831,562,0.403446,c:\workSpace\DeepLearnin_sleep\data\processed\...


,subset_name,window,split,npz_exists,feature_csv_count,feature_npz_count,summary_features,samples_npz,samples_summary,samples_index,...,X_mlp_flattened_shape,X_mlp_current_day_shape,y_mean,has_nan_X_sequence,has_inf_X_sequence,feature_count_match,sample_count_match,sequence_shape_match,mlp_flattened_shape_match,mlp_current_day_shape_match
0,daytime_context_survey,7,train,True,115,115,115,1393,1393,1393,...,"(1393, 805)","(1393, 115)",0.403446,False,False,True,True,True,True,True
1,daytime_context_survey,7,validation,True,115,115,115,269,269,269,...,"(269, 805)","(269, 115)",0.394052,False,False,True,True,True,True,True
2,daytime_context_survey,7,test,True,115,115,115,314,314,314,...,"(314, 805)","(314, 115)",0.420382,False,False,True,True,True,True,True
3,daytime_context_survey,14,train,True,115,115,115,933,933,933,...,"(933, 1610)","(933, 115)",0.382637,False,False,True,True,True,True,True
4,daytime_context_survey,14,validation,True,115,115,115,146,146,146,...,"(146, 1610)","(146, 115)",0.349315,False,False,True,True,True,True,True
5,daytime_context_survey,14,test,True,115,115,115,214,214,214,...,"(214, 1610)","(214, 115)",0.457944,False,False,True,True,True,True,True
6,daytime_context_survey,30,train,True,115,115,115,435,435,435,...,"(435, 3450)","(435, 115)",0.340230,False,False,True,True,True,True,True
7,daytime_context_survey,30,validation,True,115,115,115,55,55,55,...,"(55, 3450)","(55, 115)",0.236364,False,False,True,True,True,True,True
8,daytime_context_survey,30,test,True,115,115,115,106,106,106,...,"(106, 3450)","(106, 115)",0.594340,False,False,True,True,True,True,True
9,daytime_only,7,train,True,19,19,19,1393,1393,1393,...,"(1393, 133)","(1393, 19)",0.403446,False,False,True,True,True,True,True


problem rows: 0


,subset_name,window,split,npz_exists,feature_csv_count,feature_npz_count,summary_features,samples_npz,samples_summary,samples_index,...,X_mlp_flattened_shape,X_mlp_current_day_shape,y_mean,has_nan_X_sequence,has_inf_X_sequence,feature_count_match,sample_count_match,sequence_shape_match,mlp_flattened_shape_match,mlp_current_day_shape_match


,subset_name,window,feature_count,train_samples,any_nan,any_inf,all_feature_count_match,all_sample_count_match,all_shape_match
0,daytime_context_survey,7,115,1393,False,False,True,True,True
1,daytime_context_survey,14,115,933,False,False,True,True,True
2,daytime_context_survey,30,115,435,False,False,True,True,True
3,daytime_only,7,19,1393,False,False,True,True,True
4,daytime_only,14,19,933,False,False,True,True,True
5,daytime_only,30,19,435,False,False,True,True,True
6,full_current,7,197,1393,False,False,True,True,True
7,full_current,14,197,933,False,False,True,True,True
8,full_current,30,197,435,False,False,True,True,True
9,no_high_sleep_session,7,127,1393,False,False,True,True,True


In [29]:
# 9번. Subset-aware deep learning experiment grid 생성
# 목적:
# - 어떤 subset, window, model 조합을 학습할지 명시적으로 고정
# - full_current만 보지 않고 ablation subset을 같은 조건으로 비교
# - 너무 많은 실험을 한 번에 돌리지 않도록 단계별 실행 계획을 만든다

import pandas as pd

FEATURE_SUBSETS = [
    "full_current",
    "no_high_sleep_session",
    "no_high_sleep_session_no_stress",
    "daytime_context_survey",
    "daytime_only",
]

WINDOWS = [7, 14, 30]

MODEL_FAMILIES = [
    "mlp_current_day",
    "mlp_flattened_window",
    "simple_rnn",
    "lstm",
    "gru",
    "bilstm",
    "cnn_1d",
]

PHASE_1_MODELS = [
    "mlp_current_day",
    "gru",
    "bilstm",
    "cnn_1d",
]

PHASE_2_MODELS = [
    "mlp_flattened_window",
    "simple_rnn",
    "lstm",
]

rows = []

for subset_name in FEATURE_SUBSETS:
    for window in WINDOWS:
        for model_family in MODEL_FAMILIES:
            if model_family in PHASE_1_MODELS:
                phase = "phase_1_priority"
            else:
                phase = "phase_2_follow_up"

            rows.append({
                "phase": phase,
                "subset_name": subset_name,
                "window": window,
                "model_family": model_family,
                "run": True,
                "selection_metric": "validation_balanced_accuracy",
                "test_policy": "validation ranking 이후 확인",
            })

experiment_grid = pd.DataFrame(rows)

print("total experiments:", len(experiment_grid))
print(experiment_grid["phase"].value_counts())

display(experiment_grid)

phase_summary = (
    experiment_grid
    .groupby(["phase", "subset_name"], as_index=False)
    .agg(
        experiments=("model_family", "count"),
        windows=("window", lambda s: sorted(s.unique())),
        models=("model_family", lambda s: sorted(s.unique())),
    )
)

display(phase_summary)

total experiments: 105
phase
phase_1_priority     60
phase_2_follow_up    45
Name: count, dtype: int64


,phase,subset_name,window,model_family,run,selection_metric,test_policy
0,phase_1_priority,full_current,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
1,phase_2_follow_up,full_current,7,mlp_flattened_window,True,validation_balanced_accuracy,validation ranking 이후 확인
2,phase_2_follow_up,full_current,7,simple_rnn,True,validation_balanced_accuracy,validation ranking 이후 확인
3,phase_2_follow_up,full_current,7,lstm,True,validation_balanced_accuracy,validation ranking 이후 확인
4,phase_1_priority,full_current,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
...,...,...,...,...,...,...,...
100,phase_2_follow_up,daytime_only,30,simple_rnn,True,validation_balanced_accuracy,validation ranking 이후 확인
101,phase_2_follow_up,daytime_only,30,lstm,True,validation_balanced_accuracy,validation ranking 이후 확인
102,phase_1_priority,daytime_only,30,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
103,phase_1_priority,daytime_only,30,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인


,phase,subset_name,experiments,windows,models
0,phase_1_priority,daytime_context_survey,12,"[7, 14, 30]","[bilstm, cnn_1d, gru, mlp_current_day]"
1,phase_1_priority,daytime_only,12,"[7, 14, 30]","[bilstm, cnn_1d, gru, mlp_current_day]"
2,phase_1_priority,full_current,12,"[7, 14, 30]","[bilstm, cnn_1d, gru, mlp_current_day]"
3,phase_1_priority,no_high_sleep_session,12,"[7, 14, 30]","[bilstm, cnn_1d, gru, mlp_current_day]"
4,phase_1_priority,no_high_sleep_session_no_stress,12,"[7, 14, 30]","[bilstm, cnn_1d, gru, mlp_current_day]"
5,phase_2_follow_up,daytime_context_survey,9,"[7, 14, 30]","[lstm, mlp_flattened_window, simple_rnn]"
6,phase_2_follow_up,daytime_only,9,"[7, 14, 30]","[lstm, mlp_flattened_window, simple_rnn]"
7,phase_2_follow_up,full_current,9,"[7, 14, 30]","[lstm, mlp_flattened_window, simple_rnn]"
8,phase_2_follow_up,no_high_sleep_session,9,"[7, 14, 30]","[lstm, mlp_flattened_window, simple_rnn]"
9,phase_2_follow_up,no_high_sleep_session_no_stress,9,"[7, 14, 30]","[lstm, mlp_flattened_window, simple_rnn]"


In [30]:
# 10번. 우선 실행용 Phase 1A experiment grid 저장
# 목적:
# - 전체 105개를 한 번에 돌리지 않고, 가장 중요한 36개 실험부터 실행
# - full_current vs no_high_sleep_session vs daytime_only를 먼저 비교
# - sleep-session feature 제거 효과와 보수적 daytime-only 성능을 빠르게 확인

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "processed" / "deep_learning_subset_experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_1A_SUBSETS = [
    "full_current",
    "no_high_sleep_session",
    "daytime_only",
]

PHASE_1A_MODELS = [
    "mlp_current_day",
    "gru",
    "bilstm",
    "cnn_1d",
]

phase_1a_grid = experiment_grid[
    experiment_grid["subset_name"].isin(PHASE_1A_SUBSETS)
    & experiment_grid["model_family"].isin(PHASE_1A_MODELS)
].copy()

phase_1a_grid = phase_1a_grid.sort_values(
    ["subset_name", "window", "model_family"]
).reset_index(drop=True)

phase_1a_grid["experiment_id"] = [
    f"phase1a_{i:03d}" for i in range(len(phase_1a_grid))
]

phase_1a_grid = phase_1a_grid[
    [
        "experiment_id",
        "phase",
        "subset_name",
        "window",
        "model_family",
        "run",
        "selection_metric",
        "test_policy",
    ]
]

out_path = EXPERIMENT_DIR / "phase_1a_experiment_grid.csv"
phase_1a_grid.to_csv(out_path, index=False, encoding="utf-8-sig")

print("saved:", out_path)
print("phase_1a experiments:", len(phase_1a_grid))
display(phase_1a_grid)

phase_1a_summary = (
    phase_1a_grid
    .groupby(["subset_name", "window"], as_index=False)
    .agg(
        experiments=("experiment_id", "count"),
        models=("model_family", lambda s: sorted(s.unique())),
    )
)

display(phase_1a_summary)

saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_subset_experiments\phase_1a_experiment_grid.csv
phase_1a experiments: 36


,experiment_id,phase,subset_name,window,model_family,run,selection_metric,test_policy
0,phase1a_000,phase_1_priority,daytime_only,7,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
1,phase1a_001,phase_1_priority,daytime_only,7,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
2,phase1a_002,phase_1_priority,daytime_only,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
3,phase1a_003,phase_1_priority,daytime_only,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
4,phase1a_004,phase_1_priority,daytime_only,14,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
5,phase1a_005,phase_1_priority,daytime_only,14,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
6,phase1a_006,phase_1_priority,daytime_only,14,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
7,phase1a_007,phase_1_priority,daytime_only,14,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
8,phase1a_008,phase_1_priority,daytime_only,30,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
9,phase1a_009,phase_1_priority,daytime_only,30,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인


,subset_name,window,experiments,models
0,daytime_only,7,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
1,daytime_only,14,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
2,daytime_only,30,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
3,full_current,7,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
4,full_current,14,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
5,full_current,30,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
6,no_high_sleep_session,7,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
7,no_high_sleep_session,14,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
8,no_high_sleep_session,30,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
